# 03 · Hermes Agent from Python

[Hermes Agent](https://github.com/NousResearch/hermes-agent) (Nous Research) is written in Python, so there's no
gateway in between. We import its agent class, `run_agent.AIAgent`, and call it. It gets the same workspace as
OpenClaw:
* it reads `AGENTS.md` from its working directory;
* it opens `knowledge/*.md` with its `read_file` tool.

So any MLflow analysis that works on one runtime works on the other.

Steps: plain Hermes → Hermes plus MLflow tool spans → plus model-call spans → the same trace shape as OpenClaw
→ both runtimes side by side.

In [1]:
from sia import notebook as nb

nb.setup(db="sdk.db", experiment="acme-support-sdk")

tracking_uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db
experiment    acme-support-sdk (id 1)
backend       offline
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


{'tracking_uri': 'sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db',
 'experiment': 'acme-support-sdk (id 1)',
 'backend': 'offline',
 'agent_model': 'anthropic:/claude-haiku-5-5',
 'judge_model': 'anthropic:/claude-opus-5-5',
 'llm_judges': True}

## 1. Plain Hermes

Three things to set up first; `hermes_runtime._hermes()` does all three:
* **Isolation.** Hermes keeps config, skills, memory, and sessions in a *home* directory. We give each workspace
  its own home under `notebooks/.mlflow/hermes-sdk/` and select it with Hermes' context-local home override, so
  your `~/.hermes` is never touched.
* **Credentials.** On Claude, Hermes uses its native Anthropic provider. Hermes 0.19.0 mistakes newer Console
  API keys (`sk-ant-usr-…`) for Claude *subscription* tokens, and it can fall back to Claude Code's saved
  login. `_hermes()` makes it use `ANTHROPIC_API_KEY`, as an API key, and nothing else.
* **Autolog.** It switches OpenAI autolog off; see section 3.

One more piece of isolation: `hermes_runtime.fence(ws)` makes the workspace its own git root. Hermes tells the
model that its "Workspace Root" is the enclosing git repository. Our workspace sits inside this repo, whose
`scenario/` folder holds the reviewers' answer key, and Hermes' file tools have no hard sandbox.

In [2]:
import os
import time

from sia import config, hermes_runtime, llm

_, set_home, reset_home = hermes_runtime._hermes()
from run_agent import AIAgent                       # Hermes' agent class

ws = hermes_runtime.fence(nb.sdk_workspace("hermes"))   # its own git root: Hermes' idea of "the project"
home = hermes_runtime.home_for(ws)                  # config.yaml (+ skills) for this workspace
endpoint = llm.hermes_endpoint(config.AGENT_MODEL)
print({k: (v if k != "api_key" else "<from .env>") for k, v in endpoint.items()})
print(home)

{'model': 'claude-haiku-5-5', 'provider': 'anthropic', 'base_url': None, 'api_key': '<from .env>'}
/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/hermes-sdk/homes/hermes


In [3]:
QUESTION = "Can we set up SSO with Okta on our Pro plan?"


def run_plain(question, **callbacks):
    token, cwd = set_home(home), os.getcwd()
    os.chdir(ws)                                    # Hermes loads AGENTS.md from the working directory
    try:
        agent = AIAgent(model=endpoint["model"], provider=endpoint["provider"], base_url=endpoint["base_url"],
                        api_key=endpoint["api_key"], enabled_toolsets=["file"], skip_memory=True,
                        quiet_mode=True, max_iterations=8, max_tokens=1024, **callbacks)
        return agent.run_conversation(user_message=question)
    finally:
        os.chdir(cwd)
        reset_home(token)


t = time.perf_counter()
out = run_plain(QUESTION)
print(f"{time.perf_counter() - t:.1f}s · api_calls={out.get('api_calls')} · completed={out.get('completed')}\n")
print(out["final_response"])

11.8s · api_calls=2 · completed=True

Policies can vary by plan and region, so please double-check the Terms of Service before you plan around this.

I can't confirm from the information I have whether SSO with Okta is included in the Pro plan. My quick reference covers refunds, password resets, data export, and inviting teammates, but not SSO. I also searched the project files and found nothing about SSO, SAML, or Okta, so I don't have a reliable answer.

My best guess is that SSO is usually an enterprise-tier feature, so it may not be included in Pro. Please contact Acme Cloud sales or support to confirm whether your Pro plan supports Okta SSO, and ask about any upgrade options if it doesn't.

If you're able to use it, teammates can be added through Workspace → Members → Invite.


`run_conversation()` returns the final answer plus the whole message list, so in principle the answer and
the tool calls are all there. But there's no timing, no token usage, and nothing in MLflow.

## 2. MLflow tool spans from Hermes' callbacks

`AIAgent` takes `tool_start_callback(call_id, name, args)` and `tool_complete_callback(call_id, name, args,
result)`. Each pair becomes a TOOL span under the trace's root span. As in notebook 02, the question names the
policy file, so there are tool calls to see.

In [4]:
import mlflow
from mlflow.entities import SpanType

LOOKUP = "Please read knowledge/account_access.md and tell me: can we set up SSO with Okta on our Pro plan?"

with mlflow.start_span("acme_support_agent", span_type=SpanType.AGENT) as root:
    root.set_inputs({"question": LOOKUP})
    open_tools = {}

    def tool_start(call_id, name, args, *_, **__):
        open_tools[call_id] = mlflow.start_span_no_context(name, span_type=SpanType.TOOL, parent_span=root,
                                                           inputs=args)

    def tool_complete(call_id, name, args, result, *_, **__):
        if call_id in open_tools:
            open_tools.pop(call_id).end(outputs={"result": str(result)[:2000]})

    out = run_plain(LOOKUP, tool_start_callback=tool_start, tool_complete_callback=tool_complete)
    root.set_outputs(out["final_response"])

nb.show(nb.last_trace())

AGENT      acme_support_agent      4.2s  '**No.** According to `knowledge/account_access.md` (line 15), SSO (SAM'
  └─ TOOL       read_file               0.3s  knowledge/account_access.md


## 3. Model-call spans

The obvious move is `mlflow.anthropic.autolog()` (or `mlflow.openai.autolog()` for OpenAI-compatible providers).
But Hermes makes each model call on a worker thread, so that the call can be interrupted. Autolog spans would land
outside our trace, one disconnected trace per call.

Instead, a small subclass wraps Hermes' two model-call methods in CHAT_MODEL spans on the calling thread. On
Claude a response is an Anthropic `Message`, and its usage, including prompt-cache reads and writes, goes on
the span as MLflow's standard token-usage attribute. The subclass overrides private methods, which is why
`pyproject.toml` pins `hermes-agent==0.19.0`.

In [5]:
import inspect

print(inspect.getsource(hermes_runtime.traced_agent_class))

def traced_agent_class(base):
    """Subclass Hermes' AIAgent (or a test double) so its model calls become LLM spans."""

    class TracedAIAgent(base):
        """LLM spans on the caller's thread around Hermes' model calls."""

        _sia_parent = None

        def _interruptible_api_call(self, api_kwargs):
            return self._sia_traced(super()._interruptible_api_call, api_kwargs)

        def _interruptible_streaming_api_call(self, api_kwargs, **kw):
            parent_call = super()._interruptible_streaming_api_call
            return self._sia_traced(lambda a: parent_call(a, **kw), api_kwargs)

        def _sia_traced(self, call, api_kwargs):
            msgs = api_kwargs.get("messages") or []
            span = mlflow.start_span_no_context(
                "llm_call", span_type=SpanType.CHAT_MODEL, parent_span=self._sia_parent,
                inputs={"model": api_kwargs.get("model"), "messages": msgs[-3:], "n_messages": len(msgs)},
            )
            try:
         

In [6]:
print(nb.source(hermes_runtime, "_response_summary"))

def _response_summary(resp) -> tuple[dict, dict]:
    """Outputs and MLflow usage attributes for a Hermes model response.

    Hermes returns an OpenAI-style ChatCompletion for OpenAI-compatible providers and the Anthropic
    SDK's Message for its native Anthropic provider (Claude)."""
    usage = getattr(resp, "usage", None)
    if getattr(resp, "choices", None):                       # OpenAI-compatible
        choice = resp.choices[0]
        outputs = {"content": choice.message.content, "finish_reason": choice.finish_reason,
                   "tool_calls": [tc.function.name for tc in (choice.message.tool_calls or [])]}
        tokens = {TokenUsageKey.INPUT_TOKENS: getattr(usage, "prompt_tokens", 0) or 0,
                  TokenUsageKey.OUTPUT_TOKENS: getattr(usage, "completion_tokens", 0) or 0} if usage else {}
    else:                                                     # Anthropic Message
        blocks = getattr(resp, "content", None) or []
        outputs = {"content": "".jo

## 4. All together: `HermesAgent.ask()`

`HermesAgent` has the same interface as notebook 02's `OpenClawAgent`: `ask()`, `get_agents_md()`, and
`set_agents_md()`. Its trace has the same shape (`acme_support_agent` → `hermes.agent` → model and tool spans),
and it takes the same tags.

In [7]:
from sia.hermes_runtime import HermesAgent

nb.ensure_production()
prompt = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@production")
hermes = HermesAgent(ws)
hermes.set_agents_md(prompt.template)

answer = hermes.ask(LOOKUP, tags={"topic": "account_access", "runtime": "hermes",
                                 "prompt_version": str(prompt.version)})
trace = nb.last_trace()
nb.show(trace)
print("trace token usage:", trace.info.token_usage)

AGENT      acme_support_agent      4.1s  'Policies can vary by plan and region, so please double-check the Terms'
  └─ CHAIN      hermes.agent            4.1s  
    └─ CHAT_MODEL llm_call                0.9s  -> tools ['read_file']
    └─ TOOL       read_file               0.3s  knowledge/account_access.md
    └─ CHAT_MODEL llm_call                2.4s  -> 'Policies can vary by plan and region, so please double-check the Terms'
tags: {'topic': 'account_access', 'runtime': 'hermes', 'prompt_version': '1'}
trace token usage: {'input_tokens': 10515, 'output_tokens': 435, 'total_tokens': 10950, 'cache_read_input_tokens': 10509, 'cache_creation_input_tokens': 0}


## 5. Two runtimes, one question, one analysis

Both agents run the same `AGENTS.md` version from the Registry, against the same knowledge files, and get the
customer's question as asked. Because both leave TOOL spans with file paths, the same deterministic scorer can
say *how* each one found its answer, or that it didn't look.

In [8]:
import pandas as pd
from sia import signals
from sia.openclaw_client import Gateway, OpenClawAgent

gw = Gateway(nb.NB_STORE / "sdk" / "gateway", nb.sdk_workspace("openclaw")).start()
openclaw = OpenClawAgent(gw)
openclaw.set_agents_md(prompt.template)

rows = []
for runtime, agent in (("openclaw", openclaw), ("hermes", hermes)):
    t = time.perf_counter()
    ans = agent.ask(QUESTION, tags={"topic": "account_access", "runtime": runtime,
                                    "prompt_version": str(prompt.version)})
    tr = nb.last_trace()
    rows.append({"runtime": runtime, "seconds": round(time.perf_counter() - t, 1),
                 "files read": ", ".join(dict.fromkeys(signals.knowledge_reads(tr))) or "-",
                 "read_right_policy": signals.read_right_policy(trace=tr).value,
                 "no_disclaimer_opener": signals.no_disclaimer_opener(outputs=ans).value,
                 "tokens": (tr.info.token_usage or {}).get("total_tokens"), "answer": ans[:120]})
pd.DataFrame(rows)

,runtime,seconds,files read,read_right_policy,no_disclaimer_opener,tokens,answer
0,openclaw,7.2,-,False,True,9091,Thanks for reaching out! Policies can vary by ...
1,hermes,7.3,-,False,False,11208,"Policies can vary by plan and region, so pleas..."


## 6. Hermes' own learning loop, gated

Hermes can write its own skills and memories as it works. That's self-improvement *without* a gate. Each home's
`config.yaml` sets `write_approval: true` for both, so anything Hermes decides to save is *staged* under
`<home>/pending/` instead of going straight into the agent. Those staged writes are candidates the same MLflow
gate can approve or reject.

In [9]:
print(home / "config.yaml")
print((home / "config.yaml").read_text())

/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/hermes-sdk/homes/hermes/config.yaml
model:
  ollama_num_ctx: 65536
skills:
  write_approval: true      # Hermes' self-written skills are staged for the MLflow gate
memory:
  write_approval: true



In [10]:
openclaw.close()

**Next:** 04 · Evaluate both agents. From one trace at a time to a scorecard per agent version.